In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Singularity Machine Learning - Classification: Funkcja Qiskit od Multiverse Computing
*Zobacz [dokumentację API](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity)*


<Accordion>
<AccordionItem title="Wersje pakietów">

Kod na tej stronie został opracowany przy użyciu następujących wymagań.
Zalecamy użycie tych wersji lub nowszych.

```
scikit-learn~=1.8.0
```
</AccordionItem>
</Accordion>
> **Note:** * Funkcje Qiskit to funkcja eksperymentalna dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan oraz On-Prem (za pośrednictwem interfejsu API IBM Quantum Platform). Mają status wersji zapoznawczej i mogą ulec zmianie.

## Przegląd
Funkcja „Singularity Machine Learning - Classification" pozwala rozwiązywać rzeczywiste problemy uczenia maszynowego na sprzęcie kwantowym bez konieczności posiadania wiedzy z zakresu informatyki kwantowej. Ta funkcja aplikacyjna, oparta na metodach zespołowych, jest hybrydowym klasyfikatorem. Wykorzystuje klasyczne metody, takie jak boosting, bagging i stacking, do wstępnego trenowania zespołu. Następnie stosuje algorytmy kwantowe — wariacyjny solver własny (VQE) oraz kwantowy algorytm przybliżonej optymalizacji (QAOA) — w celu zwiększenia różnorodności wytrenowanego zespołu, jego zdolności do generalizacji oraz ogólnej złożoności.

W przeciwieństwie do innych rozwiązań z zakresu kwantowego uczenia maszynowego, ta funkcja potrafi obsługiwać wielkoskalowe zbiory danych zawierające miliony przykładów i cech, nie będąc ograniczona liczbą Qubitów w docelowym QPU. Liczba Qubitów określa jedynie rozmiar zespołu, który można wytrenować. Funkcja jest również bardzo elastyczna i może być stosowana do rozwiązywania problemów klasyfikacyjnych w szerokim zakresie dziedzin, w tym finansów, opieki zdrowotnej i cyberbezpieczeństwa.
Konsekwentnie osiąga wysoką dokładność na klasycznie trudnych problemach obejmujących wielowymiarowe, zaszumione i niezrównoważone zbiory danych.
![Jak to działa](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Przeznaczona jest dla:
1. Inżynierów i naukowców zajmujących się danymi w firmach, którzy chcą wzbogacić swoją ofertę technologiczną, integrując kwantowe uczenie maszynowe ze swoimi produktami i usługami,
2. Badaczy w laboratoriach badań kwantowych eksplorujących zastosowania kwantowego uczenia maszynowego i pragnących wykorzystać obliczenia kwantowe do zadań klasyfikacyjnych, oraz
3. Studentów i nauczycieli w instytucjach edukacyjnych, na kursach takich jak uczenie maszynowe, którzy chcą zademonstrować przewagi obliczeń kwantowych.

Poniższy przykład prezentuje różne funkcjonalności, w tym `create`, `list`, `fit` i `predict`, oraz demonstruje ich użycie na syntetycznym problemie złożonym z dwóch przeplatających się półokręgów — notorycznie trudnym ze względu na nieliniową granicę decyzyjną.


## Opis funkcji
Ta funkcja Qiskit umożliwia użytkownikom rozwiązywanie binarnych problemów klasyfikacyjnych za pomocą kwantowo ulepszonego klasyfikatora zespołowego Singularity. W tle wykorzystuje podejście hybrydowe: klasycznie trenuje zespół klasyfikatorów na oznaczonym zbiorze danych, a następnie optymalizuje go pod kątem maksymalnej różnorodności i generalizacji przy użyciu Kwantowego Algorytmu Przybliżonej Optymalizacji (QAOA) na QPU IBM&reg;. Dzięki przyjaznemu interfejsowi użytkownicy mogą skonfigurować klasyfikator zgodnie ze swoimi wymaganiami, wytrenować go na wybranym zbiorze danych i użyć do przewidywania na wcześniej niewidzianym zbiorze danych.

Aby rozwiązać ogólny problem klasyfikacyjny:
1. Przygotuj wstępnie zbiór danych i podziel go na zestawy treningowy i testowy. Opcjonalnie możesz dalej podzielić zestaw treningowy na zestawy treningowy i walidacyjny. Można to osiągnąć za pomocą [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Jeśli zestaw treningowy jest niezrównoważony, możesz go próbkować ponownie, aby wyrównać klasy, używając [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Prześlij zestawy treningowy, walidacyjny i testowy osobno do magazynu funkcji, używając metody `file_upload` katalogu i podając odpowiednią ścieżkę za każdym razem.
4. Zainicjuj klasyfikator kwantowy, używając akcji `create` funkcji, która przyjmuje hiperparametry, takie jak liczba i typy uczących się modeli, regularyzacja (wartość lambda) oraz opcje optymalizacji, w tym liczba warstw, typ klasycznego optymalizatora, Backend kwantowy itd.
5. Wytrenuj klasyfikator kwantowy na zestawie treningowym, używając akcji `fit` funkcji, podając oznaczony zestaw treningowy i zestaw walidacyjny (jeśli dotyczy).
6. Wykonaj przewidywania na wcześniej niewidzianym zestawie testowym, używając akcji `predict` funkcji.
---
## Pierwsze kroki
Uwierzytelnij się przy użyciu [klucza API IBM Quantum Platform](http://quantum.cloud.ibm.com/) i wybierz funkcję Qiskit w następujący sposób:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## Przykłady
### Klasyfikacja zbioru danych
W tym przykładzie użyjesz funkcji „Singularity Machine Learning - Classification", aby sklasyfikować zbiór danych składający się z dwóch przeplatających się, półkolistych kształtów przypominających księżyce. Zbiór danych jest syntetyczny, dwuwymiarowy i opatrzony binarnymi etykietami. Został stworzony tak, by stanowił wyzwanie dla algorytmów takich jak grupowanie oparte na centroidach czy klasyfikacja liniowa.
![Zbiór danych moons](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
Podczas tego procesu dowiesz się, jak utworzyć klasyfikator, dopasować go do danych treningowych, użyć go do przewidywania na danych testowych oraz usunąć klasyfikator po zakończeniu pracy.
Przed rozpoczęciem musisz zainstalować [scikit-learn](https://scikit-learn.org/). Zainstaluj go przy użyciu następującego polecenia:

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


Wykonaj następujące kroki:

1) Utwórz syntetyczny zbiór danych przy użyciu funkcji [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) z biblioteki [scikit-learn](https://scikit-learn.org/).
2) Prześlij wygenerowany syntetyczny zbiór danych do [wspólnego katalogu danych](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Utwórz klasyfikator wspomagany kwantowo przy użyciu akcji [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create).
4) Wypisz swoje klasyfikatory przy użyciu akcji [`list`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#list).
5) Wytrenuj klasyfikator na danych treningowych przy użyciu akcji [`fit`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#fit).
6) Użyj wytrenowanego klasyfikatora do przewidywania na danych testowych przy użyciu akcji [`predict`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#predict).
7) Usuń klasyfikator przy użyciu akcji [`delete`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#delete).
8) Posprzątaj po zakończeniu.
**Krok 1.** Zaimportuj niezbędne moduły i wygeneruj syntetyczny zbiór danych, a następnie podziel go na zbiory treningowy i testowy.

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Step 3.** Create a quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Step 4.** Train the quantum-enhanced classifier using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


**Krok 3.** Utwórz klasyfikator wspomagany kwantowo przy użyciu akcji [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create).

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


**Step 6.** Delete the quantum-enhanced classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

### `create_fit_predict` example

The following example demonstrates the [`create_fit_predict`](/docs/api/functions/multiverse-computing-singularity#create_fit_predict) action.

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Visit the [API reference](/docs/api/functions/multiverse-computing-singularity) for this Qiskit Function.
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117).

</Admonition>